# Fine-tune SpeechT5 for Text-to-Speech

This notebook demonstrates how to fine-tune Microsoft's SpeechT5 model for text-to-speech synthesis on a custom dataset. SpeechT5 is a unified-modal encoder-decoder pre-trained model that can be fine-tuned for various speech tasks including TTS.

We'll fine-tune the `microsoft/speecht5_tts` model on a custom speaker dataset to adapt it to a specific voice or domain. The process involves:
- Loading and preprocessing a TTS dataset
- Preparing text and audio features
- Creating speaker embeddings
- Training the model with Seq2Seq training
- Generating speech from the fine-tuned model

In [ ]:
!pip install transformers datasets soundfile speechbrain accelerate

## Load Dataset

We'll use a subset of a TTS dataset. For this example, we'll use the VoxPopuli dataset, but you can substitute with any TTS dataset that contains text-audio pairs.

In [ ]:
from datasets import load_dataset, Audio

# Load a TTS dataset - using LibriSpeech dummy subset as a quick example
# You can replace this with your custom dataset (e.g. VoxPopuli, LJSpeech)
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation[:20]")

# Cast audio column to Audio feature with 16kHz sampling rate
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Split into train and validation sets
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['test'])}")
print("\nExample:")
print(f"Text: {dataset['train'][0]['text']}")
print(f"Audio shape: {dataset['train'][0]['audio']['array'].shape}")
print(f"Sample rate: {dataset['train'][0]['audio']['sampling_rate']} Hz")

## Load Model and Processor

We'll load the SpeechT5 processor, the TTS model, and the HiFi-GAN vocoder for converting mel spectrograms to waveforms.

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

# Load processor and models
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

print("Models loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")

## Prepare Data

We need to prepare the data by:
1. Tokenizing the text input
2. Extracting log-mel spectrograms from audio
3. Creating speaker embeddings using SpeechBrain's pre-trained model

In [ ]:
import torch
import numpy as np
from speechbrain.pretrained import EncoderClassifier

# Load speaker encoder for creating speaker embeddings
spk_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    savedir="tmp/spk_model"
)

def create_speaker_embedding(audio_array):
    """Create speaker embedding from audio array."""
    with torch.no_grad():
        speaker_embedding = spk_model.encode_batch(torch.tensor(audio_array).unsqueeze(0))
        speaker_embedding = torch.nn.functional.normalize(speaker_embedding, dim=2)
        speaker_embedding = speaker_embedding.squeeze().cpu().numpy()
    return speaker_embedding

def prepare_dataset(example):
    """Prepare a single example for training."""
    # Tokenize text
    text = example["text"]
    example["input_ids"] = processor(text=text, return_tensors="pt").input_ids[0]
    
    # Extract log-mel spectrogram from audio
    audio = example["audio"]["array"]
    example["labels"] = processor(audio=audio, sampling_rate=16000, return_tensors="pt").input_values[0]
    
    # Create speaker embedding
    example["speaker_embeddings"] = create_speaker_embedding(audio)
    
    return example

# Apply preprocessing to dataset
print("Preprocessing dataset...")
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1  # Set to 1 to avoid multiprocessing issues with speaker model
)

print("Dataset preprocessed successfully!")
print(f"Example input_ids shape: {dataset['train'][0]['input_ids'].shape}")
print(f"Example labels shape: {dataset['train'][0]['labels'].shape}")
print(f"Example speaker_embeddings shape: {dataset['train'][0]['speaker_embeddings'].shape}")

## Data Collator

Create a custom data collator that handles padding for variable-length sequences in TTS training.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class TTSDataCollatorWithPadding:
    """Data collator for TTS that pads inputs and labels to the same length in a batch."""
    
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input_ids (text tokens)
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        batch = self.processor.pad(
            input_ids=input_ids,
            return_tensors="pt"
        )
        
        # Pad labels (mel spectrograms)
        labels = [feature["labels"] for feature in features]
        labels_max_length = max([label.shape[0] for label in labels])
        
        # Pad spectrograms
        labels_padded = []
        for label in labels:
            padding = labels_max_length - label.shape[0]
            if padding > 0:
                label = torch.nn.functional.pad(label, (0, 0, 0, padding), value=-100)
            labels_padded.append(label)
        
        batch["labels"] = torch.stack(labels_padded)
        
        # Stack speaker embeddings
        speaker_embeddings = [torch.tensor(feature["speaker_embeddings"]) for feature in features]
        batch["speaker_embeddings"] = torch.stack(speaker_embeddings)
        
        return batch

# Create data collator
data_collator = TTSDataCollatorWithPadding(processor=processor)

print("Data collator created successfully!")

## Training

Set up the training configuration and train the model using the Seq2SeqTrainer.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./speecht5_finetuned_tts",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=10,
    max_steps=50,  # Small number for demo; increase for real training
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    eval_steps=25,
    save_steps=25,
    logging_steps=10,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    greater_is_better=False,
    label_names=["labels"],
    push_to_hub=False,
)

# Initialize trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    tokenizer=processor,
)

print("Starting training...")
trainer.train()

# Save the fine-tuned model
trainer.save_model("./speecht5_finetuned_tts/final_model")
processor.save_pretrained("./speecht5_finetuned_tts/final_model")

print("Training completed and model saved!")

## Generate Speech

Now let's use the fine-tuned model to generate speech from text and listen to the results.

In [ ]:
from IPython.display import Audio
import soundfile as sf

# Load the fine-tuned model
model_finetuned = SpeechT5ForTextToSpeech.from_pretrained("./speecht5_finetuned_tts/final_model")
processor_finetuned = SpeechT5Processor.from_pretrained("./speecht5_finetuned_tts/final_model")

# Use a speaker embedding from the training data (or create a new one)
speaker_embedding = torch.tensor(dataset["train"][0]["speaker_embeddings"]).unsqueeze(0)

# Text to synthesize
text = "Hello, this is a test of the fine-tuned SpeechT5 text-to-speech model."

# Tokenize input text
inputs = processor_finetuned(text=text, return_tensors="pt")

# Generate speech
print("Generating speech...")
with torch.no_grad():
    spectrogram = model_finetuned.generate_speech(inputs["input_ids"], speaker_embedding, vocoder=vocoder)

# Convert to numpy for audio playback
speech = spectrogram.cpu().numpy()

# Save the generated speech
sf.write("generated_speech.wav", speech, samplerate=16000)
print("Speech generated and saved to 'generated_speech.wav'")

# Play the audio in the notebook
print("\nPlaying generated speech:")
Audio(speech, rate=16000)